# ETL — Tasic 2018 VISp Taxonomy (cluster reference)

Registers the **Tasic 2018 VISp scRNA-seq taxonomy** as a global cluster reference. Writes `algorithmrun/`, `clusterhierarchy/`, `cluster/`, `hierarchycategory/`. **Out of scope:** Tasic cells are not registered as `DataItem`s here.

Identifiers: `hierarchy_id="tasic_2018_visp_taxonomy"`, `run_id="tasic_2018_visp_clustering"`. Source: `anno.feather` (one row per cell, label/color columns at `class`, `subclass`, `cluster` levels). Synthetic root `cell` added so `ClusterHierarchy.root` is unique.

**Known schema caveats (documented, not fixed):**

1. **Two opposite `level` conventions.** `Cluster.level` uses `0=root` (depth from root, increasing downward). `HierarchyCategory.level` uses `0=lowest` (resolution-detail order, leaf to coarse). The two values for a given node always sum to the hierarchy depth (3 here). Don't cross-compare.
2. **`HierarchyCategory` has no taxonomy discriminator.** Categories like `class`/`subclass`/`cluster`/`major_class` are intentionally shared vocabulary across taxonomies. If a future taxonomy disagrees on a shared category's `level` or `description`, the later writer silently overwrites — so this notebook narrows its overwrite to `id IN (...)` over only the rows it owns.
3. **`HierarchyCategory.level` is generated as `Optional[str]`** (the top-level `level` slot has no `range:` set; only `Cluster.level` overrides to integer). Stored here as the str repr of the intended integer (`"0"`, `"1"`, ...). Future fix: add `range: integer` to `HierarchyCategory.level` in `slot_usage`.
4. **Color anomaly.** `Non-Neuronal` has two `class_color` values upstream (`#808285` ×577 rows, `#8D1800` ×181). Resolved by silent `dict(zip(label, color))` last-wins (matches the reference notebook). Final color: `#8D1800`.

In [1]:
from pathlib import Path

import pandas as pd
import polars as pl
import pyarrow as pa

from connects_common_connectivity.models import (
    AlgorithmRun,
    Cluster,
    ClusterHierarchy,
    HierarchyCategory,
)
from connects_common_connectivity.config import output_root
from connects_common_connectivity.io import write_models


In [2]:
INPUT_FEATHER = "/data/visp-patchseq-taxonomy-info/anno.feather"
OUTPUT_ROOT   = output_root()
HIERARCHY_ID  = "tasic_2018_visp_taxonomy"
RUN_ID        = "tasic_2018_visp_clustering"
ROOT_ID       = "cell"

assert Path(INPUT_FEATHER).exists(), f"Input not found: {INPUT_FEATHER}"
print(f"INPUT_FEATHER : {INPUT_FEATHER}")
print(f"OUTPUT_ROOT   : {OUTPUT_ROOT}")
print(f"HIERARCHY_ID  : {HIERARCHY_ID}")
print(f"RUN_ID        : {RUN_ID}")

INPUT_FEATHER : /data/visp-patchseq-taxonomy-info/anno.feather
OUTPUT_ROOT   : ../scratch/em_patchseq_wnm_v2/
HIERARCHY_ID  : tasic_2018_visp_taxonomy
RUN_ID        : tasic_2018_visp_clustering


## Load + parse `anno.feather`

In [3]:
df = pd.read_feather(INPUT_FEATHER)
print("anno.feather shape:", df.shape)

label_cols = ["class_label", "subclass_label", "cluster_label",
              "class_color", "subclass_color", "cluster_color"]
for c in label_cols:
    assert df[c].notna().all(), f"NaNs in {c}"

# Silent last-wins (matches reference notebook).
class_colors    = dict(zip(df.class_label,    df.class_color))
subclass_colors = dict(zip(df.subclass_label, df.subclass_color))
cluster_colors  = dict(zip(df.cluster_label,  df.cluster_color))

cluster_to_subclass = dict(zip(df.cluster_label, df.subclass_label))
subclass_to_class   = dict(zip(df.subclass_label, df.class_label))

# Consistency: each child has exactly one parent.
for child_col, parent_col in [("cluster_label","subclass_label"), ("subclass_label","class_label")]:
    grp = df.groupby(child_col)[parent_col].nunique()
    bad = grp[grp > 1]
    assert bad.empty, f"Multi-parent rows in {child_col}->{parent_col}: {bad.to_dict()}"

class_labels    = sorted(df.class_label.unique().tolist())
subclass_labels = sorted(df.subclass_label.unique().tolist())
cluster_labels  = sorted(df.cluster_label.unique().tolist())

# Children dicts (parent -> sorted list of children).
class_to_subclasses = {c: sorted({s for s, cl in subclass_to_class.items() if cl == c}) for c in class_labels}
subclass_to_clusters = {s: sorted({k for k, sc in cluster_to_subclass.items() if sc == s}) for s in subclass_labels}

print(f"classes={len(class_labels)}  subclasses={len(subclass_labels)}  clusters={len(cluster_labels)}")
print(f"Non-Neuronal class_color resolved to: {class_colors['Non-Neuronal']}")

anno.feather shape: (14236, 152)


classes=3  subclasses=23  clusters=111
Non-Neuronal class_color resolved to: #8D1800


## `HierarchyCategory` — 4 rows (`major_class`/`class`/`subclass`/`cluster`)

In [4]:
# HierarchyCategory.level uses 0=lowest (resolution-detail order, leaf to coarse).
# stored as str (schema gap noted above).
category_rows = [
    HierarchyCategory(id="cluster",     description="Leaf cluster (cell type / T-type).",        level="0"),
    HierarchyCategory(id="subclass",    description="Subclass of cell types.",                   level="1"),
    HierarchyCategory(id="class",       description="Top-level transcriptomic class.",           level="2"),
    HierarchyCategory(id="major_class", description="Synthetic root grouping all classes.",      level="3"),
]
CATEGORY_IDS = [c.id for c in category_rows]

result = write_models(category_rows)
print(f"HierarchyCategory written: {result.rows_written} rows")


HierarchyCategory written: 4 rows


In [5]:
verify_cat = pl.read_delta(OUTPUT_ROOT + "hierarchycategory/").filter(pl.col("id").is_in(CATEGORY_IDS))
print(verify_cat.shape)
assert verify_cat.shape[0] == 4
print(verify_cat.sort("id"))

(4, 3)
shape: (4, 3)
┌─────────────┬─────────────────────────────────┬───────┐
│ id          ┆ description                     ┆ level │
│ ---         ┆ ---                             ┆ ---   │
│ str         ┆ str                             ┆ str   │
╞═════════════╪═════════════════════════════════╪═══════╡
│ class       ┆ Top-level transcriptomic class… ┆ 2     │
│ cluster     ┆ Leaf cluster (cell type / T-ty… ┆ 0     │
│ major_class ┆ Synthetic root grouping all cl… ┆ 3     │
│ subclass    ┆ Subclass of cell types.         ┆ 1     │
└─────────────┴─────────────────────────────────┴───────┘


## `AlgorithmRun` — 1 row

In [6]:
run_row = AlgorithmRun(
    id=RUN_ID,
    algorithm_name="hierarchical (Tasic et al. 2018, VISp scRNA-seq taxonomy)",
    algorithm_version="2018",
    score_description=None,
    distance_description=None,
    # input_dataset intentionally omitted: Tasic cells are not registered in this codebase.
    # produced_hierarchies omitted: schema declares it as inlined dict[id, ClusterHierarchy].
    # The inverse ClusterHierarchy.run carries the link; storing inlined hierarchies in the
    # algorithmrun/ row would duplicate state. Future: schema flip to inlined: false (list of ids).
)

result = write_models([run_row])
print(f"AlgorithmRun written: {result.rows_written} rows")


AlgorithmRun written: 1 rows


In [7]:
verify_run = pl.read_delta(OUTPUT_ROOT + "algorithmrun/").filter(pl.col("id") == RUN_ID)
print(verify_run.shape)
assert verify_run.shape[0] == 1
print(verify_run)

(1, 9)
shape: (1, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ algorithm ┆ algorithm ┆ json_obje ┆ … ┆ input_dat ┆ produced_ ┆ score_des ┆ distance │
│ ---       ┆ _name     ┆ _version  ┆ ct        ┆   ┆ aset      ┆ hierarchi ┆ cription  ┆ _descrip │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ es        ┆ ---       ┆ tion     │
│           ┆ str       ┆ str       ┆ str       ┆   ┆ str       ┆ ---       ┆ str       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆ str       ┆           ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ tasic_201 ┆ hierarchi ┆ 2018      ┆ null      ┆ … ┆ null      ┆ null      ┆ null      ┆ null     │
│ 8_visp_cl ┆ cal       ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ ustering  ┆ (Tasic et ┆           ┆           ┆   ┆           ┆     

## `Cluster` — 138 rows (1 synthetic root + 3 classes + 23 subclasses + 111 leaf clusters)

In [8]:
cluster_rows: list[Cluster] = []

# Synthetic root: hierarchy_category=major_class, level=0 (depth from root).
cluster_rows.append(Cluster(
    id=ROOT_ID,
    hierarchy_id=HIERARCHY_ID,
    parent=None,
    children=class_labels,
    level=0,
    hex_color="#000000",
    hierarchy_category="major_class",
))

# Class level: depth 1.
for cl in class_labels:
    cluster_rows.append(Cluster(
        id=cl,
        hierarchy_id=HIERARCHY_ID,
        parent=ROOT_ID,
        children=class_to_subclasses[cl],
        level=1,
        hex_color=class_colors[cl],
        hierarchy_category="class",
    ))

# Subclass level: depth 2.
for sc in subclass_labels:
    cluster_rows.append(Cluster(
        id=sc,
        hierarchy_id=HIERARCHY_ID,
        parent=subclass_to_class[sc],
        children=subclass_to_clusters[sc],
        level=2,
        hex_color=subclass_colors[sc],
        hierarchy_category="subclass",
    ))

# Cluster (leaf) level: depth 3.
for k in cluster_labels:
    cluster_rows.append(Cluster(
        id=k,
        hierarchy_id=HIERARCHY_ID,
        parent=cluster_to_subclass[k],
        children=[],
        level=3,
        hex_color=cluster_colors[k],
        hierarchy_category="cluster",
    ))

assert len(cluster_rows) == 1 + len(class_labels) + len(subclass_labels) + len(cluster_labels)
print(f"Cluster rows built: {len(cluster_rows)}")
result = write_models(cluster_rows)
print(f"Cluster written: {result.rows_written} rows")

Cluster rows built: 138


Cluster written: 138 rows


In [9]:
verify_clu = pl.read_delta(OUTPUT_ROOT + "cluster/").filter(pl.col("hierarchy_id") == HIERARCHY_ID)
print(verify_clu.shape)
assert verify_clu.shape[0] == 1 + 3 + 23 + 111 == 138
assert verify_clu.filter(pl.col("id") == ROOT_ID).shape[0] == 1
assert set(verify_clu.select("hierarchy_category").to_series().to_list()) == {"major_class","class","subclass","cluster"}
print(verify_clu.group_by("hierarchy_category").len().sort("hierarchy_category"))

(138, 9)
shape: (4, 2)
┌────────────────────┬─────┐
│ hierarchy_category ┆ len │
│ ---                ┆ --- │
│ str                ┆ u32 │
╞════════════════════╪═════╡
│ class              ┆ 3   │
│ cluster            ┆ 111 │
│ major_class        ┆ 1   │
│ subclass           ┆ 23  │
└────────────────────┴─────┘


## `ClusterHierarchy` — 1 row

In [10]:
hierarchy_row = ClusterHierarchy(
    id=HIERARCHY_ID,
    run=RUN_ID,
    root=ROOT_ID,
    clusters=[c.id for c in cluster_rows],
)
result = write_models([hierarchy_row])
print(f"ClusterHierarchy written: {result.rows_written} rows")

ClusterHierarchy written: 1 rows


In [11]:
verify_h = pl.read_delta(OUTPUT_ROOT + "clusterhierarchy/").filter(pl.col("id") == HIERARCHY_ID)
print(verify_h.shape)
assert verify_h.shape[0] == 1
row = verify_h.row(0, named=True)
assert row["root"] == ROOT_ID
assert row["run"] == RUN_ID
assert len(row["clusters"]) == 138
print(f"root={row['root']}  run={row['run']}  clusters={len(row['clusters'])}")

(1, 4)
root=cell  run=tasic_2018_visp_clustering  clusters=138


## Summary

Written under `../scratch/em_patchseq_wnm_v1/`:

| Table | Rows | Predicate |
|---|---|---|
| `algorithmrun/` | 1 | `id = RUN_ID` |
| `clusterhierarchy/` | 1 | `id = HIERARCHY_ID` |
| `cluster/` | 138 | `hierarchy_id = HIERARCHY_ID` (partitioned) |
| `hierarchycategory/` | 4 | `id IN (...)` |

Tasic taxonomy: 1 synthetic root + 3 classes + 23 subclasses + 111 leaf clusters. No `DataItem`s registered (out of scope). Idempotent.